# Test Adaptive Alpha Repair
This notebook tests the new `AdaptiveAlphaCalculator` integrated into the `WeightedVCRepairer`.

In [3]:
import pandas as pd
import numpy as np
import os
import sys
import json
sys.path.append('..')

from src.loading.dcs_loader import load_dcs
from src.entities.dataset import Dataset
from src.synthesizing.model_loader import SmartNoiseModelLoader
from src.marginals_obtaining.top_k_obtainer import TopKObtainer
from src.marginals_obtaining.utility_functions.distance_utility import DistanceUtility
from src.repairing.weighted_vc_repairer import WeightedVCRepairer
from src.utils.mbi_patch import apply_patch

## 1. Setup Data and Marginals

In [4]:
apply_patch()

dataset_name = "adult"
data = pd.read_csv(f"../data/{dataset_name}/data.csv").head(1000) # Small sample for quick test
dcs = load_dcs(f"../data/{dataset_name}/dcs.txt")

with open(f"../data/{dataset_name}/metadata.json") as f:
    metadata = json.load(f)
target = metadata['target']

ds = Dataset(name=dataset_name, data=data, dcs=dcs, target=target)

model_path = f"../models/{dataset_name}_aim.pkl"
if os.path.exists(model_path):
    loader = SmartNoiseModelLoader(model_path=model_path, seed=42)
    sds = loader.synthesize(ds)
    
    obtainer = TopKObtainer(1.0, 1.0, 5, DistanceUtility())
    marginals = obtainer.obtain(ds, sds)
    print(f"Obtained {len(marginals)} marginals.")

Applying reproducibility patch to mbi.graphical_model.GraphicalModel...
Generating 1000 rows using model from ..\models\adult_aim.pkl (seed=42)...


c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\utils\mbi_patch.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(list(proj), group_keys=False).apply(foo)
c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\utils\mbi_patch.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(list(proj), group_keys=False).apply(foo)
c:\Users\itayc\OneDrive\Desktop\Master\final

Obtained 5 marginals.


c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\marginals_obtaining\top_k_obtainer.py:68: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  idx = p_counts.index.union(s_counts.index)
c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\marginals_obtaining\top_k_obtainer.py:68: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  idx = p_counts.index.union(s_counts.index)
c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\marginals_obtaining\top_k_obtainer.py:68: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  idx = p_counts.index.union(s_counts.index)
c:\Users\itayc\OneDrive\Desktop\Master\final\notebooks\..\src\marginals_obtaining\top_k_obtainer.py:68: RuntimeWarning: The values in the array are unorderable. Pass `sort=False` to suppress this warning.
  idx = p_counts.index.union(s_counts.index)


## 2. Run Repair with Adaptive Alpha

In [5]:
if 'sds' in locals():
    print("Running repair with fixed alpha=0.5...")
    repairer_fixed = WeightedVCRepairer(alpha=0.5, use_adaptive_alpha=False)
    res_fixed = repairer_fixed.repair(sds, marginals)
    
    print("\nRunning repair with adaptive alpha...")
    repairer_adaptive = WeightedVCRepairer(alpha=0.5, use_adaptive_alpha=True)
    res_adaptive = repairer_adaptive.repair(sds, marginals)
    
    print(f"\nFixed Alpha Deletions: {len(sds.data) - len(res_fixed.data)}")
    print(f"Adaptive Alpha Deletions: {len(sds.data) - len(res_adaptive.data)}")

Running repair with fixed alpha=0.5...

Running repair with adaptive alpha...

Fixed Alpha Deletions: 566
Adaptive Alpha Deletions: 566
